In [0]:
%sql
-- CREATE VOLUME IF NOT EXISTS workspace.default.sensor_to_s3_checkpoints;

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType

# Step 1 — read from Kafka
df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "13.222.103.18:9092")
    .option("subscribe", "sensor-readings")
    .option("startingOffsets", "earliest")
    .load()
)

raw_readings = df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)", "timestamp")

# Step 2 — parse JSON into structured columns
schema = (
    StructType()
    .add("sensor_id", StringType())
    .add("sensor_type", StringType())
    .add("timestamp", StringType())
    .add("value", DoubleType())
    .add("unit", StringType())
    .add("location", StringType())
    .add("battery_level", DoubleType())
)

bronze_df = raw_readings.select(
    from_json(col("value"), schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select("data.*", "kafka_timestamp")

# Step 3 — write to Bronze Delta table
(
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/workspace/default/sensor_to_s3_checkpoints/bronze")
    .trigger(availableNow=True)
    .toTable("sensor_bronze")
)

In [0]:
%sql
SELECT COUNT(*) FROM sensor_bronze;

In [0]:
from pyspark.sql.functions import col, to_timestamp

bronze = spark.read.table("sensor_bronze")

silver = (
    bronze
    .filter(col("sensor_id").isNotNull())
    .filter(col("value").isNotNull())
    .filter(col("location").isNotNull())
    .filter(
        ((col("sensor_type") == "temperature") & (col("value").between(-10, 40))) |
        ((col("sensor_type") == "humidity") & (col("value").between(0, 100))) |
        ((col("sensor_type") == "vibration") & (col("value").between(0, 10)))
    )
    .withColumn("event_timestamp", to_timestamp(col("timestamp")))
    .drop("timestamp")
    .dropDuplicates(["sensor_id", "event_timestamp"])
)

silver.write.format("delta").mode("overwrite").saveAsTable("sensor_silver")

display(silver)

In [0]:
from pyspark.sql.functions import avg, count, min, max, window, col

silver = spark.read.table("sensor_silver")

gold = (
    silver
    .groupBy(
        "sensor_id",
        "sensor_type",
        "location",
        window(col("event_timestamp"), "5 minutes")
    )
    .agg(
        avg("value").alias("avg_value"),
        min("value").alias("min_value"),
        max("value").alias("max_value"),
        count("*").alias("reading_count")
    )
    .select(
        "sensor_id", "sensor_type", "location",
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "avg_value", "min_value", "max_value", "reading_count"
    )
)

gold.write.format("delta").mode("overwrite").saveAsTable("sensor_gold")

display(gold)

In [0]:
import boto3
import os

# Write Gold to a Volume path (confirmed working earlier)
gold = spark.read.table("sensor_gold")
local_volume_path = "/Volumes/workspace/default/sensor_to_s3_checkpoints/gold_temp"
gold.write.format("delta").mode("overwrite").save(local_volume_path)

# Upload to S3 using boto3
s3_client = boto3.client(
    "s3",
    aws_access_key_id="Insert Here",
    aws_secret_access_key="Insert here",
    region_name="insert Here",
)

bucket_name = "sensor-to-s3-data-lake-amara-2026"

uploaded_count = 0
for root, dirs, files in os.walk(local_volume_path):
    for file in files:
        local_file = os.path.join(root, file)
        s3_key = f"gold/sensor_gold/{os.path.relpath(local_file, local_volume_path)}"
        s3_client.upload_file(local_file, bucket_name, s3_key)
        uploaded_count += 1

print(f"Uploaded {uploaded_count} files to s3://{bucket_name}/gold/sensor_gold/")